# EDSS 취업데이터 2022–2023 단과대학·학과 교차검증

## tl;dr

기존 학과명 기반 잠정 매칭 66개를 `단과대학명 + 학과명` 조합으로 다시 검증했다. 62개가 같은 2022 개방ID로 재확인됐고 4개는 단과대학 조합 중첩이 기준보다 낮아 검토 대상으로 내려갔다. 새로 수락되거나 다른 개방ID로 변경된 매칭은 없었다.

## Context & Methods

- 비교 집단: 2023 대학과 일반대학원 600개 학교 단위
- 후보 필터: 같은 지역·구분, 고유 학과 수 차이 0~2
- 우선 순위: 단과대학–학과 조합 교집합, Jaccard, 작은 집합 포괄률을 학과명 단독 지표보다 먼저 적용
- 수락: 단과대학–학과 조합의 정확 일치(3개 이상) 또는 Jaccard 0.80 이상·포괄률 0.90 이상, 양방향 유일 최상위
- 결과는 연구용 잠정 교차표이며 정식 패널에는 기록하지 않음

### Key Assumptions

단과대학 명칭은 연도 사이에 개편될 수 있으므로 기준 미달은 오매칭 확정이 아니라 검토 필요로 해석한다.

In [1]:
from pathlib import Path
import csv
import hashlib
import json
import subprocess
import pandas as pd

ROOT = Path('/Users/joocheol/Documents/ChatGPT/EDSS')
RAW_ROOT = Path('/Users/joocheol/Documents/GitHub/edss/data/raw/edss')
SCRIPT = ROOT / 'scripts/build_edss_employment_2022_2023_college_department_crosswalk.py'
SUMMARY = ROOT / 'data/metadata/edss_employment_2022_2023_college_department_crosswalk.json'
REVIEW = ROOT / 'data/processed/edss/restricted/derived/employment_2022_2023_college_department_review.csv'
CROSSWALK = ROOT / 'data/processed/edss/restricted/derived/employment_2022_2023_college_department_crosswalk.csv'

## Data

원본 두 개를 읽어 분석을 다시 실행하고 체크섬과 필수 열 완전성을 기록한다.

In [2]:
run = subprocess.run(
    ['python3', str(SCRIPT), '--raw-root', str(RAW_ROOT)],
    cwd=ROOT, check=True, capture_output=True, text=True,
)
summary = json.loads(SUMMARY.read_text(encoding='utf-8'))
pd.DataFrame(summary['data_quality']).T

,blank_college_count,blank_department_count,blank_open_id_count,profile_count,row_count,blank_province_count,blank_school_name_count
source_2022,0.0,0.0,0.0,537.0,553521.0,NaN,NaN
source_2023,0.0,0.0,NaN,1619.0,23312.0,0.0,0.0


## Results

식별값은 출력하지 않고 집계와 기준 미달 학교명만 표시한다.

In [3]:
pd.DataFrame(summary['counts']['scope_status_counts']).fillna(0).astype(int).T

,accepted_reciprocal_exact_college_department_set,accepted_reciprocal_high_college_department_overlap,ambiguous_forward_pair_score,no_department_count_tolerance_candidate,review_small_exact_college_department_signature,review_weak_college_department_overlap
대학,28,22,15,55,14,281
일반대학원,4,8,7,46,10,110


In [4]:
pd.Series(summary['counts']['comparison_to_department_only_counts'], name='학교 수').rename_axis('기존 결과와 비교').to_frame()

,학교 수
기존 결과와 비교,
not_accepted,534
previous_match_confirmed_by_college_department,62
previous_match_downgraded_by_college_department,4


In [5]:
with REVIEW.open(encoding='utf-8-sig', newline='') as handle:
    review_rows = list(csv.DictReader(handle))
downgraded = [r for r in review_rows if r['comparison_outcome'] == 'previous_match_downgraded_by_college_department']
pd.DataFrame([{
    '지역': r['province'],
    '학교명': r['school_name_2023'],
    '학과명 Jaccard': float(r['department_jaccard']),
    '단과대학–학과 Jaccard': float(r['college_department_pair_jaccard']),
    '단과대학–학과 포괄률': float(r['college_department_pair_smaller_set_coverage']),
} for r in downgraded])

,지역,학교명,학과명 Jaccard,단과대학–학과 Jaccard,단과대학–학과 포괄률
0,경기,아주대학교,0.947368,0.761905,0.888889
1,서울,추계예술대학교,1.000000,0.571429,0.727273
2,서울,한국외국어대학교,0.810345,0.746032,0.886792
3,인천,인천대학교,0.833333,0.747368,0.855422


### Validation

새 교차표가 기존 잠정 결과의 동일한 매핑 62개만 포함하는지 검증한다.

In [6]:
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

with CROSSWALK.open(encoding='utf-8-sig', newline='') as handle:
    crosswalk_rows = list(csv.DictReader(handle))
assert len(review_rows) == summary['counts']['review_identity_count'] == 600
assert len(crosswalk_rows) == summary['counts']['accepted_pair_validated_count'] == 62
assert len(downgraded) == 4
assert summary['counts']['comparison_to_department_only_counts'].get('previous_match_changed_by_college_department', 0) == 0
assert summary['counts']['comparison_to_department_only_counts'].get('new_match_from_college_department_disambiguation', 0) == 0
assert len({(r['scope'], r['province'], r['school_name_2023']) for r in crosswalk_rows}) == len(crosswalk_rows)
assert len({(r['scope'], r['province'], r['open_id_2022']) for r in crosswalk_rows}) == len(crosswalk_rows)
assert all(r['pair_match_status'].startswith('accepted_') for r in crosswalk_rows)
assert all(r['canonical_mapping_written'] == 'false' for r in crosswalk_rows)
assert sha256(REVIEW) == summary['outputs']['review']['sha256']
assert sha256(CROSSWALK) == summary['outputs']['crosswalk']['sha256']
{'review_rows': len(review_rows), 'pair_validated_rows': len(crosswalk_rows), 'downgraded': len(downgraded), 'validation': 'PASS'}

{'review_rows': 600,
 'pair_validated_rows': 62,
 'downgraded': 4,
 'validation': 'PASS'}

## Takeaways

- 단과대학–학과 조합은 기존 66개 중 62개를 같은 개방ID로 재확인했다.
- 대학은 50개, 일반대학원은 12개가 조합 기준을 통과했다.
- 아주대학교, 추계예술대학교, 한국외국어대학교, 인천대학교는 학과명 증거는 강하지만 단과대학 조합이 기준보다 낮아 검토 대상으로 유지한다.
- 새 매칭과 ID 변경은 0건이므로 단과대학 정보는 후보 확대보다 기존 결과의 정밀 검증에 기여했다.